# 💍 Jewellery Studio: Qwen-Image-Edit-2511 on free Kaggle (2× T4)

Takes a model photo and a jewellery product photo, and makes a photo of the model wearing that jewellery.
Comes with a page that works like the Hugging Face Space (Edit tab and Batch tab) and gives you a public `gradio.live` link.

**Kaggle settings:** Accelerator **GPU T4 x2** · Internet **On** · Secret **`HF_TOKEN`** (ticked) · dataset added through **Add Input**.
In the dataset, put model photos with `model` in the name (`0_model.png`, `1_model.png`), and use any name for the jewellery photos
(`necklace_01.jpg`, `earings_01.jpg`, ...).

**Order:** Cell 1, then **Run → Restart & clear cell outputs**, then Cells 2 → 5. After any restart, run Cells 2 → 5 again. Cell 1 isn't needed after a restart.

Tested on 2× T4: about **9.5 min per image** at the default sizes (8 steps).

## 1. Install (once per session, then RESTART)

In [ ]:
# Versions that work together (checked against PyPI). diffusers 0.39 supports Qwen-Image-Edit-2511.
!pip install -q "diffusers==0.39.0" "transformers==4.57.1" "huggingface_hub>=0.34,<1.0" "gradio==5.50.0" accelerate peft bitsandbytes safetensors sentencepiece
!pip uninstall -y torchao        # Kaggle's old torchao blocks LoRA loading
import importlib.metadata as m
for p in ["diffusers", "transformers", "huggingface_hub", "gradio", "bitsandbytes", "peft", "accelerate"]:
    print(p, m.version(p))
print("\n👉 Now: Run → Restart & clear cell outputs, then run Cell 2")

## 2. HF token + bitsandbytes fix

In [ ]:
import os, importlib
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
print("✅ HF logged in as:", whoami()["name"])

# On Kaggle, diffusers sometimes fails to detect bitsandbytes. Tell it explicitly.
import bitsandbytes as bnb
import diffusers.utils.import_utils as d_iu
d_iu._bitsandbytes_available = True
d_iu._bitsandbytes_version = bnb.__version__
import transformers.utils.import_utils as t_iu
if hasattr(t_iu, "_bitsandbytes_available"):
    t_iu._bitsandbytes_available = True
for name in ["diffusers.quantizers.bitsandbytes.utils", "diffusers.quantizers.bitsandbytes.bnb_quantizer",
             "transformers.integrations.bitsandbytes", "transformers.quantizers.quantizer_bnb_4bit"]:
    try: importlib.import_module(name).bnb = bnb
    except Exception: pass
from diffusers.utils import is_bitsandbytes_available
print("✅ bitsandbytes visible to diffusers:", is_bitsandbytes_available())

## 3. Load the model on both GPUs (~3–4 min, ~33 GB download the first time)
* **GPU 0:** image model (4-bit), layers 0–35
* **GPU 1:** image model layers 36–59, text encoder (original, 4-bit, vision part kept at full precision) and VAE

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import glob, math, gc, time, random, shutil
import torch, torch.nn.functional as F
from PIL import Image, ImageOps
from accelerate.hooks import add_hook_to_module, AlignDevicesHook
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from diffusers import (QwenImageEditPlusPipeline, QwenImageTransformer2DModel,
                       AutoencoderKLQwenImage, FlowMatchEulerDiscreteScheduler)
import diffusers.pipelines.qwenimage.pipeline_qwenimage_edit_plus as qp

REPO_4BIT = "ovedrive/Qwen-Image-Edit-2511-4bit"   # image model (4-bit)
REPO_ORIG = "Qwen/Qwen-Image-Edit-2511"            # text encoder + VAE (original files)
OUTPUT_DIR = "/kaggle/working/outputs"; os.makedirs(OUTPUT_DIR, exist_ok=True)
GPU_IMG, GPU_TXT = torch.device("cuda:0"), torch.device("cuda:1")
N_MOVE = 24              # image-model layers moved to GPU 1 (20 worked; 24 leaves more room on GPU 0)
REF_PX = 768             # size the input photos are processed at (pipeline default 1024 = out of memory on T4)

assert torch.cuda.device_count() >= 2, "❌ Need GPU T4 x2 (Settings → Accelerator)"

def health(model, label):
    empty = [n for n, p in model.named_parameters() if p.device.type == "meta"]
    devs = sorted({str(p.device) for p in model.parameters()})
    print(f"   {label}: on {devs} | empty weights: {len(empty)}")
    assert not empty, f"❌ {label} has empty weights, e.g. {empty[:3]}"

def mem():
    return f"GPU0 {torch.cuda.memory_allocated(0)/1e9:.1f} GB | GPU1 {torch.cuda.memory_allocated(1)/1e9:.1f} GB"

# T4 has no fast bf16 attention -> run only the attention op in fp16
if not getattr(F, "_patched", False):
    _orig = F.scaled_dot_product_attention
    _h = lambda t: t.half() if torch.is_tensor(t) and t.dtype == torch.bfloat16 else t
    def _sdpa(query, key, value, *args, **kwargs):
        if query.dtype == torch.bfloat16 and query.is_cuda:
            return _orig(_h(query), _h(key), _h(value), *[_h(a) for a in args],
                         **{n: _h(v) for n, v in kwargs.items()}).to(torch.bfloat16)
        return _orig(query, key, value, *args, **kwargs)
    F.scaled_dot_product_attention = _sdpa; F._patched = True

print("1/4 image model …")
torch.cuda.set_device(0)
transformer = QwenImageTransformer2DModel.from_pretrained(REPO_4BIT, subfolder="transformer", torch_dtype=torch.bfloat16)
health(transformer, "transformer"); print("  ", mem())

print("2/4 text encoder (4-bit, vision part full precision) …")
text_encoder = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    REPO_ORIG, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map={"": 1},
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual"]))
health(text_encoder, "text_encoder"); print("  ", mem())

print("3/4 VAE …")
vae = AutoencoderKLQwenImage.from_pretrained(REPO_ORIG, subfolder="vae", torch_dtype=torch.bfloat16).to(GPU_TXT)
health(vae, "vae")
pipe = QwenImageEditPlusPipeline.from_pretrained(
    REPO_4BIT, transformer=transformer, text_encoder=text_encoder, vae=vae, torch_dtype=torch.bfloat16)

print("4/4 Lightning 8-step LoRA …")
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config({
    "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False, "max_image_seq_len": 8192,
    "max_shift": math.log(3), "num_train_timesteps": 1000, "shift": 1.0, "shift_terminal": None,
    "stochastic_sampling": False, "time_shift_type": "exponential", "use_beta_sigmas": False,
    "use_dynamic_shifting": True, "use_exponential_sigmas": False, "use_karras_sigmas": False})
try:
    pipe.load_lora_weights("lightx2v/Qwen-Image-Edit-2511-Lightning",
                           weight_name="Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors")
    DEFAULT_STEPS, DEFAULT_CFG = 8, 1.0
    print("   ✅ Lightning loaded")
except Exception as e:
    print("   ⚠️ Lightning failed, normal mode (much slower):", repr(e))
    pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(REPO_ORIG, subfolder="scheduler")
    DEFAULT_STEPS, DEFAULT_CFG = 30, 4.0

# ---- device wiring ----
QwenImageEditPlusPipeline._execution_device = property(lambda self: GPU_TXT)   # pipeline runs on GPU 1
add_hook_to_module(pipe.transformer, AlignDevicesHook(execution_device=GPU_IMG, io_same_device=True))
blocks = pipe.transformer.transformer_blocks
for i in range(len(blocks) - N_MOVE, len(blocks)):                                # last layers -> GPU 1
    blocks[i].to(GPU_TXT)
    add_hook_to_module(blocks[i], AlignDevicesHook(execution_device=GPU_TXT, io_same_device=False))
for mod in [pipe.transformer.norm_out, pipe.transformer.proj_out]:                # output layers on GPU 0
    add_hook_to_module(mod, AlignDevicesHook(execution_device=GPU_IMG, io_same_device=False))
qp.VAE_IMAGE_SIZE = REF_PX * REF_PX
try: pipe.vae.enable_tiling()
except Exception: pass
pipe.set_progress_bar_config(disable=False)

gc.collect(); torch.cuda.empty_cache()
health(pipe.transformer, "transformer (split over 2 GPUs)")
print("\npipeline device:", pipe._execution_device, "|", mem())
print(f"✅ Model ready | steps={DEFAULT_STEPS}")

## 4. Dataset, prompts, helpers + test ONE image (~10 min)

In [ ]:
RUN_TEST = True          # set False to skip the test image

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
DATA_IMGS  = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.lower().endswith(IMG_EXT))
MODEL_IMGS = [p for p in DATA_IMGS if "model" in os.path.basename(p).lower()]
JEWEL_IMGS = [p for p in DATA_IMGS if p not in MODEL_IMGS]
name = lambda p: os.path.splitext(os.path.basename(p))[0]
print("👩 models   :", [name(p) for p in MODEL_IMGS])
print("💍 jewellery:", len(JEWEL_IMGS), "images")

_KEEP = ("Do not change her face, identity, expression, hairstyle, skin, clothes, pose, lighting or background. "
         "Ignore the background and any logo or watermark of Picture 2. ")
PROMPTS = {
 "necklace": ("Picture 1 is a photo of a woman. Picture 2 is a product photo of a gold necklace / mangalsutra. "
    "Make the woman in Picture 1 wear this exact necklace from Picture 2 around her neck, with the pendant hanging at the "
    "centre of her chest just below the collarbone. Copy the design exactly: same pendant shape, engraving and enamel, same "
    "chain style, and the same black beads and gold beads if Picture 2 has them. 22k yellow gold, realistic delicate "
    "true-to-life size, natural gold reflections, the chain resting naturally on her skin and over her clothes. " + _KEEP +
    "Do not add earrings or any other jewellery. Photorealistic jewellery photoshoot, sharp focus on the necklace."),
 "earrings": ("Picture 1 is a photo of a woman. Picture 2 is a product photo of gold earrings. Put this exact pair of "
    "earrings on her earlobes (on each visible ear; if hair covers an ear the earring may be partly hidden). Copy the design "
    "exactly: same shape, gold colour, stones and beads, realistic small true-to-life size, natural metallic shine. " + _KEEP +
    "Do not add a necklace or any other jewellery. Photorealistic jewellery photoshoot, sharp focus on the earrings."),
 "ring": ("Picture 1 is a photo of a woman. Picture 2 is a product photo of a gold ring. Put this exact ring on the ring "
    "finger of her visible hand. Same design, stones and gold finish, realistic size. " + _KEEP + "Photorealistic jewellery photoshoot."),
 "bangles": ("Picture 1 is a photo of a woman. Picture 2 is a product photo of gold bangles. Put these exact bangles on her "
    "visible wrist. Same design and gold finish, realistic size. " + _KEEP + "Photorealistic jewellery photoshoot."),
}
DEFAULT_PROMPT = PROMPTS["necklace"]

def guess_type(path):
    f = os.path.basename(path).lower()
    if "ear" in f:                       return "earrings"     # earring_ / earings_
    if "ring" in f:                      return "ring"
    if "bangle" in f or "bracelet" in f: return "bangles"
    return "necklace"                                           # necklace_ / neckalce_ / mangalsutra_

def load_rgb(p): return ImageOps.exif_transpose(Image.open(p)).convert("RGB")

def size_for(img, area):
    w, h = img.size; s = math.sqrt(area / (w * h))              # keep Picture 1's shape
    return round(w*s/16)*16, round(h*s/16)*16

def run(images, prompt, seed, steps=None, area=768*960, ref_px=REF_PX, n=1):
    qp.VAE_IMAGE_SIZE = int(ref_px) * int(ref_px)
    W, H = size_for(images[0], area)
    with torch.inference_mode():
        out = pipe(image=images, prompt=prompt, negative_prompt=" ", width=W, height=H,
                   num_inference_steps=int(steps or DEFAULT_STEPS), true_cfg_scale=DEFAULT_CFG,
                   num_images_per_prompt=int(n), generator=torch.Generator("cpu").manual_seed(int(seed))).images
    gc.collect(); torch.cuda.empty_cache()
    return out

def side_by_side(*imgs, h=512):
    imgs = [i.resize((int(i.width*h/i.height), h)) for i in imgs]
    c = Image.new("RGB", (sum(i.width for i in imgs), h), "white"); x = 0
    for i in imgs: c.paste(i, (x, 0)); x += i.width
    return c

if RUN_TEST:
    m = load_rgb(MODEL_IMGS[0])
    jp = next((p for p in JEWEL_IMGS if name(p) == "necklace_01"), JEWEL_IMGS[0])
    j = load_rgb(jp)
    torch.cuda.reset_peak_memory_stats(0); torch.cuda.reset_peak_memory_stats(1)
    t = time.time()
    out = run([m, j], PROMPTS[guess_type(jp)], 0)[0]
    print(f"⏱ {time.time()-t:.0f} s | peak GPU0 {torch.cuda.max_memory_allocated(0)/1e9:.1f} GB, "
          f"GPU1 {torch.cuda.max_memory_allocated(1)/1e9:.1f} GB")
    out.save(f"{OUTPUT_DIR}/test.png")
    display(side_by_side(j, m, out))

## 5. The Jewellery Studio page (public link)
Open the `https://xxxx.gradio.live` link it prints. Keep the Kaggle tab open while you use it.
**Download your ZIP before you stop the session**, because `/kaggle/working` is cleared when an interactive session ends.

In [ ]:
import gradio as gr
MAX_SEED = 2**31 - 1
OUT_SIZES = {"Small 640×800 (fastest)": 640*800, "Medium 768×960 (tested)": 768*960, "Large 896×1120 (may run out of memory)": 896*1120}
REF_SIZES = {"512 (fastest)": 512, "640": 640, "768 (tested)": 768}

def to_pils(items):
    out = []
    for it in items or []:
        x = it[0] if isinstance(it, (list, tuple)) else it
        if isinstance(x, dict): x = x.get("path") or x.get("name")
        out.append(x.convert("RGB") if isinstance(x, Image.Image) else load_rgb(x))
    return out

def fmt(sec): return f"{int(sec//60)} min {int(sec%60)} s"

# ---------------- Tab 1: like the HF Space ----------------
def infer(images, prompt, seed, randomize, steps, out_size, ref_size, n):
    imgs = to_pils(images)
    if len(imgs) < 2: raise gr.Error("Add 2 images: 1st = model photo, 2nd = jewellery photo")
    if randomize: seed = random.randint(0, MAX_SEED)
    t = time.time()
    try:
        out = run(imgs, prompt, seed, steps, OUT_SIZES[out_size], REF_SIZES[ref_size], n)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise gr.Error("Out of GPU memory. Choose a smaller output size or input-photo size")
    paths = []
    for k, im in enumerate(out):                      # save as PNG and show the file (avoids a Gradio WebP bug)
        pth = f"{OUTPUT_DIR}/edit_{int(time.time())}_{k}.png"
        im.convert("RGB").save(pth); paths.append(pth)
    return paths, seed, f"✅ done in {fmt(time.time()-t)} · seed {seed} · saved {paths[0]}"

# ---------------- Tab 2: batch ----------------
def batch(model_choice, model_upload, jewel_names, extra_files, auto_prompt, prompt, extra_text,
          seed, randomize, steps, out_size, ref_size, skip):
    if model_upload is not None:
        models = [("uploaded", model_upload.convert("RGB"))]
    elif model_choice == "All models":
        models = [(name(p), load_rgb(p)) for p in MODEL_IMGS]
    else:
        models = [(model_choice, load_rgb(next(p for p in MODEL_IMGS if name(p) == model_choice)))]
    jewels = [p for p in JEWEL_IMGS if name(p) in (jewel_names or [])]
    jewels += [f if isinstance(f, str) else f.name for f in (extra_files or [])]
    if not models or not jewels: raise gr.Error("Pick at least one model and one jewellery image")

    outdir = f"{OUTPUT_DIR}/batch"; os.makedirs(outdir, exist_ok=True)
    jobs = [(mn, mi, jp) for mn, mi in models for jp in jewels]
    gallery, done_t, t0 = [], [], time.time()
    for k, (mn, mi, jp) in enumerate(jobs, 1):
        out = f"{outdir}/{name(jp)}__{mn}.png"
        if not (skip and os.path.exists(out)):
            p = (PROMPTS[guess_type(jp)] if auto_prompt else prompt) + (" " + extra_text.strip() if extra_text.strip() else "")
            s = random.randint(0, MAX_SEED) if randomize else int(seed)
            t = time.time()
            try:
                run([mi, load_rgb(jp)], p, s, steps, OUT_SIZES[out_size], REF_SIZES[ref_size])[0].convert("RGB").save(out)
                done_t.append(time.time() - t)
            except Exception as e:
                print("failed", out, repr(e)); gc.collect(); torch.cuda.empty_cache(); continue
        gallery = (gallery + [(out, f"{name(jp)} · {mn}")])[-24:]
        eta = (sum(done_t)/len(done_t)) * (len(jobs) - k) if done_t else 0
        yield gallery, None, f"⏳ {k}/{len(jobs)} done · elapsed {fmt(time.time()-t0)} · about {fmt(eta)} left"
    zip_path = shutil.make_archive(f"{OUTPUT_DIR}/jewellery_results", "zip", outdir)
    yield gallery, zip_path, f"✅ Finished {len(jobs)} images in {fmt(time.time()-t0)}. Download the ZIP below"

def speed_settings():
    with gr.Accordion("Speed / quality settings", open=False):
        steps = gr.Slider(4, 12, value=DEFAULT_STEPS, step=1, label="Steps (8 = Lightning default; fewer = faster, rougher)")
        with gr.Row():
            out_size = gr.Dropdown(list(OUT_SIZES), value="Medium 768×960 (tested)", label="Output size")
            ref_size = gr.Dropdown(list(REF_SIZES), value="768 (tested)", label="Input-photo processing size")
        with gr.Row():
            seed = gr.Slider(0, MAX_SEED, value=0, step=1, label="Seed")
            rnd = gr.Checkbox(value=True, label="Randomize seed")
    return steps, out_size, ref_size, seed, rnd

with gr.Blocks(title="Jewellery Studio") as demo:
    gr.Markdown("# 💍 Jewellery Studio · Qwen-Image-Edit-2511\n"
                "Free Kaggle 2×T4. **About 9–10 min per image** at the tested settings. Smaller sizes are faster.")

    with gr.Tab("Edit (like HF Space)"):
        with gr.Row():
            inp = gr.Gallery(label="Input: 1st = model photo, 2nd = jewellery photo", type="pil", interactive=True, height=450)
            res = gr.Gallery(label="Result", type="filepath", format="png", height=450)
        prompt = gr.Textbox(value=DEFAULT_PROMPT, label="Prompt", lines=4)
        with gr.Row():
            for t in PROMPTS:
                gr.Button(f"Use {t} prompt", size="sm").click(lambda t=t: PROMPTS[t], None, prompt)
        steps, out_size, ref_size, seed, rnd = speed_settings()
        n = gr.Slider(1, 2, value=1, step=1, label="Number of images (2 = double time)")
        btn = gr.Button("✨ Edit!", variant="primary")
        info = gr.Markdown()
        btn.click(infer, [inp, prompt, seed, rnd, steps, out_size, ref_size, n], [res, seed, info])

    with gr.Tab("Batch (model × jewellery)"):
        gr.Markdown(f"Dataset: **{len(MODEL_IMGS)} models**, **{len(JEWEL_IMGS)} jewellery**. "
                    "Results are saved to `/kaggle/working/outputs/batch` (already-done images are skipped).")
        with gr.Row():
            bmodel_choice = gr.Dropdown(["All models"] + [name(p) for p in MODEL_IMGS],
                                        value=name(MODEL_IMGS[0]) if MODEL_IMGS else "All models", label="Model photo")
            bmodel_up = gr.Image(type="pil", label="…or upload another model photo", height=220)
        bjewels = gr.CheckboxGroup([name(p) for p in JEWEL_IMGS], value=[name(p) for p in JEWEL_IMGS],
                                   label="Jewellery to generate (untick to skip)")
        bextra = gr.File(file_count="multiple", file_types=["image"], type="filepath", label="Extra jewellery photos (optional)")
        bauto = gr.Checkbox(value=True, label="Auto prompt from file name (earings_ → earrings, necklace_ → necklace, ring_, bangle_)")
        bprompt = gr.Textbox(value=DEFAULT_PROMPT, label="Prompt (used when auto prompt is OFF)", lines=3)
        bextra_text = gr.Textbox(label="Extra text added to every prompt (optional)", placeholder="e.g. make the pendant a little smaller")
        bsteps, bout, bref, bseed, brnd = speed_settings()
        bskip = gr.Checkbox(value=True, label="Skip images already generated (resume)")
        bbtn = gr.Button("✨ Generate all", variant="primary")
        bstatus = gr.Textbox(label="Status")
        bgal = gr.Gallery(label="Latest results", columns=4, format="png", height=500)
        bzip = gr.File(label="Download all (ZIP)")
        bbtn.click(batch, [bmodel_choice, bmodel_up, bjewels, bextra, bauto, bprompt, bextra_text,
                           bseed, brnd, bsteps, bout, bref, bskip], [bgal, bzip, bstatus])

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)

## Troubleshooting
| Problem | Fix |
|---|---|
| `CUDA out of memory` | Choose a smaller **Output size** or **Input-photo size** on the page |
| `FLAX_WEIGHTS_NAME` / import errors | You didn't restart after Cell 1. Restart, then run Cells 2 → 5 |
| `❌ … empty weights` in Cell 3 | Restart and run Cell 3 again. If it repeats, report the output |
| Jewellery design not exact | Use input-photo size 768, 8–10 steps, a tightly cropped product photo, and the "Extra text" box |
| Link stopped working | The session ended or went idle. Run Cells 2 → 5 again (no downloads needed) |